# 06 LMArena leaderboard: human-preference ratings

**Purpose.** This notebook downloads the public LMArena leaderboard dataset (text arena, latest split) and saves it as `lmarena_ratings.csv`. Q1 uses it as a second, independent view of model quality: ECI measures benchmark skill, while LMArena measures how people vote when they compare answers side by side.

**Question served.** Q1 (a second quality signal) and Q2.

**How to run.** Run all cells from top to bottom. The download needs access to `huggingface.co`. If it is blocked, follow the manual route in Section 4.

## At a glance: links, authorisation, download, timing

| Item | Detail |
|---|---|
| Kind | Download of one Parquet file (text arena, latest snapshot) |
| Where it comes from | LMArena on Hugging Face. Dataset page https://huggingface.co/datasets/lmarena-ai/leaderboard-dataset . Folder https://huggingface.co/datasets/lmarena-ai/leaderboard-dataset/tree/main/text . File `latest-00000-of-00001.parquet`. Live leaderboard https://lmarena.ai |
| Licence and authorisation | cc-by-4.0 (dataset page, checked 2026-10-18). Public dataset, no login or token. Credit LMArena. Details in Section 1 |
| Download size | about 614 KB (the `full` file is about 60 MB and is not used) |
| Expected time | download 5 to 30 seconds, whole notebook under 1 minute. Measured in our test run: 3.1 seconds from the hand-saved file with 11,033 rows. The online download could not be tested from the test machine |
| Data as of | the snapshot date in the file (2026-10-15 in the copy tested). See `processed/data_currency.csv` |
| To update | Run the notebook again, or save the new Parquet file by hand |

## 1. Source, reference links and authorisation

| Item | Detail |
|---|---|
| Publisher | LMArena (lmarena-ai) |
| Dataset | Leaderboard dataset, text arena configuration |
| Dataset page | https://huggingface.co/datasets/lmarena-ai/leaderboard-dataset |
| Live leaderboard (for comparison by eye) | https://lmarena.ai |
| How the notebook finds the file | The Hugging Face dataset viewer API lists the dataset's Parquet files: https://datasets-server.huggingface.co/parquet?dataset=lmarena-ai/leaderboard-dataset . API documentation: https://huggingface.co/docs/dataset-viewer/parquet |
| Licence | The dataset page shows the licence field `cc-by-4.0`. Checked 18 Oct 2026. Licence text: https://creativecommons.org/licenses/by/4.0/ . Credit LMArena as the source. |
| Authorisation | The dataset is public and not gated. No account, token or login is used. The notebook makes a handful of requests. |
| Citation | LMArena, leaderboard-dataset, Hugging Face, retrieved on the date in `raw/retrieval_log.json`. Check the dataset page for the paper or citation text the authors prefer |
| Assumption to confirm | The configuration name `text` and the split name `latest` come from the dataset page as we read it on 18 Oct 2026. Section 4 prints every configuration and split it finds, so a rename is visible at once |

## 2. Table structure

### 2.1 Structure of the source (text configuration)

One row per model and per leaderboard category. Columns as listed on the dataset page:

| Source column | Type | Meaning |
|---|---|---|
| `model_name` | text | the name used by LMArena |
| `organization` | text | developer |
| `license` | text | licence label used by LMArena (for example proprietary or an open licence). This is LMArena's label and may differ from the licence text, as with Epoch's label |
| `rating` | float | Bradley-Terry style Elo rating, higher is better |
| `rating_lower`, `rating_upper` | float | bounds of the confidence interval |
| `variance` | float | variance of the rating estimate |
| `vote_count` | int | number of votes behind the rating |
| `rank` | int | rank within the category |
| `category` | text | the leaderboard view, for example overall or a task category |
| `leaderboard_publish_date` | date | the date of the snapshot |

Splits: `full` (all snapshots) and `latest` (the newest snapshot). We use `latest`.

### 2.2 Structure of the output `processed/lmarena_ratings.csv`

Same columns as above, with these changes: `leaderboard_publish_date` is parsed to a date, `is_overall` is added (true where the category is the overall view) and rows are sorted by category and rank. Columns missing in the source are left out and listed by the notebook.

## 3. Set up the paths and helpers

In [1]:
import hashlib
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

T0 = time.time()
FOLDER = "06_lmarena_leaderboard"
cwd = Path.cwd()
if cwd.name == FOLDER:
    HERE = cwd
elif (cwd / FOLDER).exists():
    HERE = cwd / FOLDER
elif (cwd / "Data" / FOLDER).exists():
    HERE = cwd / "Data" / FOLDER
else:
    raise FileNotFoundError(f"Open this notebook from the folder {FOLDER} (or from Data/). Current folder: {cwd}")

DATA = HERE.parent               # the Data folder
MANUAL = DATA / "raw" / "lmarena"                # hand-downloaded or pre-copied source files
RAW = HERE / "raw"               # files this notebook downloads, kept exactly as received
OUT = HERE / "processed"         # tables this notebook writes
for folder in (RAW, OUT):
    folder.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAW / "retrieval_log.json"
print("Notebook folder:", HERE)
print("Manual folder  :", MANUAL)


HEADERS = {"User-Agent": "UTS-36104-A3-student-project (academic use, no contact data stored)"}
TIMEOUT = 120
REFRESH = True   # True: try a fresh download first. False: reuse the file already in raw/ if there is one.


def record(source_id: str, url: str, path: Path, how: str) -> None:
    """Add (or replace) the retrieval-log entry for one raw file."""
    content = path.read_bytes()
    log = json.loads(LOG_PATH.read_text(encoding="utf-8")) if LOG_PATH.exists() else {}
    log[source_id] = {
        "url": url,
        "file": path.name,
        "how": how,
        "retrieved_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "bytes": len(content),
        "sha256": hashlib.sha256(content).hexdigest(),
    }
    LOG_PATH.write_text(json.dumps(log, indent=2), encoding="utf-8")
    print(f"logged {source_id}: {len(content):,} bytes via {how}")


def get_file(source_id: str, url: str, target: str, manual_name: str = None) -> Path:
    """Return the path of a raw file. Order: fresh download, earlier download in raw/, manual copy."""
    path = RAW / target
    if REFRESH or not path.exists():
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
            path.write_bytes(response.content)
            record(source_id, url, path, "downloaded")
            return path
        except Exception as err:
            print(f"Download of {source_id} failed: {err}")
    manual = MANUAL / (manual_name or target)
    candidates = [p for p in (path, manual) if p.exists()]
    if candidates:
        best = max(candidates, key=lambda p: p.stat().st_mtime)   # the newest copy wins
        if best == path:
            print(f"Using the earlier download {path.name}")
        else:
            shutil.copy2(best, path)
            record(source_id, url, path, f"copied from manual folder {manual.parent.name}")
        return path
    raise FileNotFoundError(
        f"Could not get {source_id}. Open {url} in a browser, save the file as '{manual_name or target}' "
        f"inside {MANUAL}, then run this cell again.")


CONFIG = "text"
SPLIT = "latest"
PARQUET_API = "https://datasets-server.huggingface.co/parquet?dataset=lmarena-ai/leaderboard-dataset"

Notebook folder: d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\06_lmarena_leaderboard
Manual folder  : d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\raw\lmarena


## 4. Acquire the data

Route A downloads the Parquet file(s) through the dataset viewer API. Route B reads a file you saved by hand into `Data/raw/lmarena/` (any `.parquet` or `.csv`). Manual steps for route B: open the dataset page above, click the Files tab, open the `text` folder, download the Parquet file for the `latest` split and save it in `Data/raw/lmarena/`.

In [2]:
def download_parquet_files():
    """Route A. Returns a list of local Parquet paths, or an empty list if anything fails."""
    try:
        listing = requests.get(PARQUET_API, headers=HEADERS, timeout=TIMEOUT)
        listing.raise_for_status()
        files = listing.json().get("parquet_files", [])
    except Exception as err:
        print("Could not list the Parquet files:", err)
        return []
    pairs = sorted({(f["config"], f["split"]) for f in files})
    print("Configurations and splits found:", pairs)
    wanted = [f for f in files if f["config"] == CONFIG and f["split"] == SPLIT]
    if not wanted:
        print(f"No files for config '{CONFIG}' and split '{SPLIT}'. Change CONFIG and SPLIT above to one of the pairs printed.")
        return []
    local = []
    for i, f in enumerate(wanted):
        target = RAW / f"lmarena_{CONFIG}_{SPLIT}_{i}.parquet"
        try:
            r = requests.get(f["url"], headers=HEADERS, timeout=TIMEOUT)
            r.raise_for_status()
            target.write_bytes(r.content)
            record(f"lmarena_{CONFIG}_{SPLIT}_{i}", f["url"], target, "downloaded")
            local.append(target)
        except Exception as err:
            print("Download failed:", f["url"], err)
            return []
    return local


def manual_files():
    """Route B. Files saved by hand in Data/raw/lmarena/."""
    found = sorted(list(MANUAL.glob("*.parquet")) + list(MANUAL.glob("*.csv"))) if MANUAL.exists() else []
    for path in found:
        record("lmarena_manual_" + path.stem, "manual file, see Section 4", path, "manual file in Data/raw/lmarena")
    return found


def read_any(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)


sources = []
if REFRESH:
    sources = download_parquet_files()
if not sources:
    # No fresh download. Take the newest file among the earlier download and the hand-saved files.
    pool = sorted(RAW.glob("lmarena_*.parquet")) + manual_files()
    if pool:
        newest = max(pool, key=lambda p: p.stat().st_mtime)
        print("Using the newest saved file:", newest)
        sources = [newest]
if not sources:
    raise FileNotFoundError(
        "No LMArena file found. Save the Parquet file for config 'text', split 'latest' from "
        "https://huggingface.co/datasets/lmarena-ai/leaderboard-dataset into " + str(MANUAL) + " and run this cell again.")

arena_raw = pd.concat([read_any(p) for p in sources], ignore_index=True)
print("Source files:", [p.name for p in sources])
print("Rows and columns:", arena_raw.shape)
arena_raw.head()

Configurations and splits found: [('agent', 'full'), ('agent', 'latest'), ('agent_bash_recovery_steps', 'full'), ('agent_bash_recovery_steps', 'latest'), ('agent_bash_reliability', 'full'), ('agent_bash_reliability', 'latest'), ('agent_bash_reliability_no_prereq', 'full'), ('agent_bash_reliability_no_prereq', 'latest'), ('agent_praise_complaint', 'full'), ('agent_praise_complaint', 'latest'), ('agent_steerability', 'full'), ('agent_steerability', 'latest'), ('agent_task_outcome_explicit', 'full'), ('agent_task_outcome_explicit', 'latest'), ('agent_tool_hallucination', 'full'), ('agent_tool_hallucination', 'latest'), ('document', 'full'), ('document', 'latest'), ('document_style_control', 'full'), ('document_style_control', 'latest'), ('image_edit', 'full'), ('image_edit', 'latest'), ('image_to_video', 'full'), ('image_to_video', 'latest'), ('search', 'full'), ('search', 'latest'), ('search_factuality', 'full'), ('search_factuality', 'latest'), ('search_style_control', 'full'), ('search

,model_name,organization,license,rating,rating_lower,rating_upper,variance,vote_count,rank,category,leaderboard_publish_date
0,Gemini 4 Argon (High),google,Proprietary,1532.337544,1523.557533,1541.117554,20.067527,4904.0,1.0,overall,2026-10-15
1,Claude Opus 5.5 (High),anthropic,Proprietary,1515.761445,1508.427927,1523.094963,14.000016,8300.0,2.0,overall,2026-10-15
2,Claude Fable 5.1 (Max),anthropic,Proprietary,1511.553459,1505.512521,1517.594397,9.499759,14590.0,3.0,overall,2026-10-15
3,Claude Opus 5 (Max),anthropic,Proprietary,1507.636187,1502.834012,1512.438363,6.003159,32495.0,4.0,overall,2026-10-15
4,Claude Opus 5 (High),anthropic,Proprietary,1502.538235,1498.880290,1506.196180,3.483198,66824.0,5.0,overall,2026-10-15


## 5. Inspect

In [3]:
expected = ["model_name", "organization", "license", "rating", "rating_lower", "rating_upper",
            "variance", "vote_count", "rank", "category", "leaderboard_publish_date"]
missing_cols = [c for c in expected if c not in arena_raw.columns]
extra_cols = [c for c in arena_raw.columns if c not in expected]
print("Columns missing against Section 2:", missing_cols)
print("Extra columns in the source:", extra_cols)
assert "model_name" in arena_raw.columns and "rating" in arena_raw.columns, "LMArena changed the structure. Update Section 2."
print()
print(arena_raw.dtypes)
print()
print("Missing values per column:")
print(arena_raw.isna().sum())
if "category" in arena_raw.columns:
    print()
    print("Categories:", sorted(arena_raw["category"].dropna().unique())[:40])
if "leaderboard_publish_date" in arena_raw.columns:
    print("Snapshot dates:", sorted(arena_raw["leaderboard_publish_date"].astype(str).unique())[-5:])

Columns missing against Section 2: []
Extra columns in the source: []

model_name                   object
organization                 object
license                      object
rating                      float64
rating_lower                float64
rating_upper                float64
variance                    float64
vote_count                  float64
rank                        float64
category                     object
leaderboard_publish_date     object
dtype: object

Missing values per column:
model_name                  0
organization                0
license                     0
rating                      0
rating_lower                0
rating_upper                0
variance                    0
vote_count                  0
rank                        0
category                    0
leaderboard_publish_date    0
dtype: int64

Categories: ['chinese', 'coding', 'creative_writing', 'english', 'exclude_ties', 'expert', 'french', 'german', 'hard_prompts', 'hard_prompts_englis

## 6. Transform

If the file holds several snapshot dates, only the newest is kept. Ratings and counts are converted to numbers. The `is_overall` flag marks the all-purpose view, which is the right default for Q1.

In [4]:
arena = arena_raw.copy()
if "leaderboard_publish_date" in arena.columns:
    arena["leaderboard_publish_date"] = pd.to_datetime(arena["leaderboard_publish_date"], errors="coerce")
    newest = arena["leaderboard_publish_date"].max()
    print("Newest snapshot:", newest.date() if pd.notna(newest) else "unknown")
    arena = arena[arena["leaderboard_publish_date"] == newest]

for col in ["rating", "rating_lower", "rating_upper", "variance", "vote_count", "rank"]:
    if col in arena.columns:
        arena[col] = pd.to_numeric(arena[col], errors="coerce")

if "category" in arena.columns:
    arena["is_overall"] = arena["category"].astype(str).str.lower().isin(["overall", "default", "text_overall"])
    print("Rows flagged as overall:", int(arena["is_overall"].sum()))
else:
    arena["is_overall"] = True
    print("No category column. Treating every row as overall.")

sort_cols = [c for c in ["category", "rank"] if c in arena.columns]
arena = arena.sort_values(sort_cols).reset_index(drop=True)
arena.head()

Newest snapshot: 2026-10-15
Rows flagged as overall: 416


,model_name,organization,license,rating,rating_lower,rating_upper,variance,vote_count,rank,category,leaderboard_publish_date,is_overall
0,Gemini 4 Argon (High),google,Proprietary,1608.306550,1576.907642,1639.705457,256.645050,363.0,1.0,chinese,2026-10-15,False
1,Claude Fable 5.1 (Max),anthropic,Proprietary,1585.640069,1567.150724,1604.129413,88.991156,1111.0,2.0,chinese,2026-10-15,False
2,Claude Opus 5.5 (High),anthropic,Proprietary,1583.569316,1561.533555,1605.605077,126.403742,767.0,3.0,chinese,2026-10-15,False
3,Claude Opus 5 (Max),anthropic,Proprietary,1573.386192,1560.778889,1585.993495,41.375971,2580.0,4.0,chinese,2026-10-15,False
4,Claude Opus 5 (High),anthropic,Proprietary,1567.827409,1558.611645,1577.043173,22.108869,5347.0,5.0,chinese,2026-10-15,False


## 7. Validate

In [5]:
problems = []
if arena["rating"].isna().any():
    problems.append("missing ratings")
key = [c for c in ["model_name", "category"] if c in arena.columns]
if arena.duplicated(subset=key).any():
    problems.append("duplicated model_name within a category")
if "rank" in arena.columns and (arena["rank"] < 1).any():
    problems.append("rank below 1")
if "vote_count" in arena.columns and (arena["vote_count"] < 0).any():
    problems.append("negative vote_count")
if {"rating_lower", "rating_upper"} <= set(arena.columns):
    ci = arena.dropna(subset=["rating_lower", "rating_upper"])
    if not ((ci["rating_lower"] <= ci["rating"]) & (ci["rating"] <= ci["rating_upper"])).all():
        problems.append("rating outside its interval")
print("Problems:", problems if problems else "none")
assert not problems, problems

print()
print("Rows:", len(arena), "| models:", arena["model_name"].nunique())
overall = arena[arena["is_overall"]]
if len(overall):
    cols = [c for c in ["model_name", "organization", "license", "rating", "vote_count", "rank"] if c in overall.columns]
    print("Top 10 in the overall view:")
    print(overall.sort_values("rating", ascending=False)[cols].head(10).to_string(index=False))

Problems: none

Rows: 11033 | models: 416
Top 10 in the overall view:
             model_name organization     license      rating  vote_count  rank
  Gemini 4 Argon (High)       google Proprietary 1532.337544      4904.0   1.0
 Claude Opus 5.5 (High)    anthropic Proprietary 1515.761445      8300.0   2.0
 Claude Fable 5.1 (Max)    anthropic Proprietary 1511.553459     14590.0   3.0
    Claude Opus 5 (Max)    anthropic Proprietary 1507.636187     32495.0   4.0
   Claude Opus 5 (High)    anthropic Proprietary 1502.538235     66824.0   5.0
 Claude Opus 4.6 (High)    anthropic Proprietary 1501.694398     81394.0   6.0
      Muse Wonder (Max)         meta Proprietary 1500.955479      3247.0   7.0
Gemini 3.8 Flash (High)       google Proprietary 1498.882344     37454.0   8.0
        Claude Opus 4.6    anthropic Proprietary 1497.546141     86071.0   9.0
  Claude Fable 5 (High)    anthropic Proprietary 1491.226586     42004.0  10.0


## 8. Save

In [6]:
arena.to_csv(OUT / "lmarena_ratings.csv", index=False, encoding="utf-8")
print("saved", OUT / "lmarena_ratings.csv", arena.shape)

saved d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\06_lmarena_leaderboard\processed\lmarena_ratings.csv (11033, 12)


## 9. Quick look: organisations in the overall view

Which developers appear and how LMArena labels their licences. Use it to see which China-based models have enough votes to trust. It is a sanity view and not a result.

In [7]:
if "organization" in overall.columns and len(overall):
    view = (overall.groupby("organization")
            .agg(models=("model_name", "nunique"),
                 best_rating=("rating", "max"),
                 total_votes=("vote_count", "sum") if "vote_count" in overall.columns else ("rating", "size"))
            .sort_values("best_rating", ascending=False))
    display(view.head(15))
else:
    print("No organisation column or no overall rows to summarise.")

,models,best_rating,total_votes
organization,,,
google,41,1532.337544,1691698.0
anthropic,33,1515.761445,2166770.0
meta,23,1500.955479,779775.0
xiaomi,8,1490.678501,243371.0
alibaba,42,1482.877225,1056206.0
moonshot,7,1474.664592,258046.0
zai,16,1471.456030,371817.0
openai,48,1471.339766,2285886.0
baidu,4,1467.729221,90514.0


## How to update, and what to do if the download is blocked

**Automatic.** Run all cells again. Section 4 asks the Hugging Face dataset viewer for the file list and downloads the `text` / `latest` file. About 10 seconds.

**Manual download (use this if you see `Could not list the Parquet files`).**

1. Open https://huggingface.co/datasets/lmarena-ai/leaderboard-dataset/tree/main/text in a browser.
2. In the list, click the download arrow on the row **latest-00000-of-00001.parquet** (about 614 KB). Do not take the `full` file.
3. Direct address for the same file: https://huggingface.co/datasets/lmarena-ai/leaderboard-dataset/resolve/main/text/latest-00000-of-00001.parquet
4. Save it into `Data\raw\lmarena\`. Any `.parquet` name works.
5. Run the notebook again. With no fresh download, it takes the **newest** file among the earlier download in `raw/` and the files in `Data/raw/lmarena/`.

**Update check.** The snapshot date printed in Section 5 and `data_covers_to` in `processed/data_currency.csv` should match the date shown on the dataset page (the folder listing showed "Update text for 2026-10-15" on 2026-10-18). After any update, open `raw/retrieval_log.json` in this folder. The `how` field says `downloaded` when the fresh file was used and `copied from manual folder` when the hand-saved copy was used. Then open `processed/data_currency.csv` and confirm `extracted_utc` and `data_covers_to` are what you expect.

## 10. Run time and limits

**Time to extract.** Route A makes one small request for the file list and then one download per Parquet file. The text configuration is a small table (thousands of rows), so expect **5 to 30 seconds** in total on a normal connection. Reading and saving takes a few seconds. The last cell prints the measured time. If you use route B, the run is under 10 seconds.

**Limits and known risks.**

1. This is a snapshot of human votes. Votes come from people who chose to use the site, so the population is not a random sample of enterprise users.
2. Style effects can lift a rating. Treat LMArena as a second view next to ECI, not as truth.
3. `license` is LMArena's label, not the licence text.
4. Model names differ from Epoch and LiteLLM. The crosswalk is a separate step.
5. The configuration and split names are assumptions confirmed only by the printed list in Section 4.
6. This notebook could not be run from the build environment because `huggingface.co` was not reachable there. The logic was tested with a small stand-in Parquet file in route B. The first real run on your computer is the real test.

## 11. References

1. LMArena, leaderboard-dataset. https://huggingface.co/datasets/lmarena-ai/leaderboard-dataset
2. LMArena, live leaderboard. https://lmarena.ai
3. Hugging Face, dataset viewer Parquet API. https://huggingface.co/docs/dataset-viewer/parquet
4. Creative Commons, Attribution 4.0 International. https://creativecommons.org/licenses/by/4.0/
5. Project note on dataset licences: `05 Gemini Deep Research/Licence Check for Open-Weight Models (English and 繁體中文).md`

## Data currency: how up to date is this data?

Every notebook writes `processed/data_currency.csv` with the day it was extracted, the dates the data itself covers, and what we know about when the source started and how often it changes. Use these values for the "data as of" label in the app.

In [8]:
def write_currency(dataset, covers_from, covers_to, snapshot_note, origin_note, frequency):
    """Record how current the data is. extracted_utc is the moment this notebook ran."""
    row = {
        "dataset": dataset,
        "extracted_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "data_covers_from": str(covers_from)[:10] if covers_from is not None else "not stated in the data",
        "data_covers_to": str(covers_to)[:10] if covers_to is not None else "not stated in the data",
        "snapshot_note": snapshot_note,
        "source_origin_note": origin_note,
        "source_update_frequency": frequency,
    }
    pd.DataFrame([row]).to_csv(OUT / "data_currency.csv", index=False, encoding="utf-8")
    for key, value in row.items():
        print(f"{key:24s} {value}")


snap_date = arena["leaderboard_publish_date"].max() if "leaderboard_publish_date" in arena.columns else None
write_currency("LMArena text leaderboard (lmarena_ratings.csv)",
               snap_date, snap_date,
               "latest split. The data is one leaderboard snapshot, so it covers the single publish date shown. Votes behind it accumulate over time before that date",
               "Start date of LMArena votes not recorded in our notes. Check the dataset page.",
               "Leaderboard snapshots are published periodically, the dataset page shows the latest")

dataset                  LMArena text leaderboard (lmarena_ratings.csv)
extracted_utc            2026-10-17T22:31:37Z
data_covers_from         2026-10-15
data_covers_to           2026-10-15
snapshot_note            latest split. The data is one leaderboard snapshot, so it covers the single publish date shown. Votes behind it accumulate over time before that date
source_origin_note       Start date of LMArena votes not recorded in our notes. Check the dataset page.
source_update_frequency  Leaderboard snapshots are published periodically, the dataset page shows the latest


In [9]:
elapsed = time.time() - T0
print(f"Elapsed time of this run: {elapsed:.1f} seconds")
print("Files written:")
for p in sorted(OUT.glob("*")):
    print(f"  {p.relative_to(HERE)}  ({p.stat().st_size:,} bytes)")

Elapsed time of this run: 7.7 seconds
Files written:
  processed\data_currency.csv  (520 bytes)
  processed\lmarena_ratings.csv  (1,784,938 bytes)
